# Inference

Batch-scores eligible clients at an inference date, looping a configurable list of products, and writes one **long-format** table `(cif, product, model_hash, infer_date, score)` to an output path.

It does **not** retrain. Each product is scored with its own **frozen config + saved model + calibration offset** from `artifacts/<product>/<run_id>/`, so results are reproducible and independent of any later YAML edits. Eligibility comes from the frozen config, so only *eligible* clients are scored. If the model was trained with downsampling, the stored calibration is re-applied automatically.

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / 'configs' / '_schema.py').exists())
sys.path.insert(0, str(ROOT))

from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

from src.inference import discover_products, resolve_run, load_scorer, score_product

### Parameters — everything configurable here
- `PRODUCTS`: `None` scores **all** products that have a trained model; or set a list.
- `INFER_DATE`: `None` uses each product's frozen inference date; or override for all (e.g. `'2026-08-01'`) to score a new month.
- `RUN_ID`: `None` uses the **latest** run per product; or pin a specific `run_id`.
- `OUTPUT_PATH`: single long-format destination, partitioned by product.

In [ ]:
ART_ROOT    = str(ROOT / 'artifacts')                 # where trained models live
OUTPUT_PATH = 'hdfs:///retail/scores/propensity_long'
INFER_DATE  = None        # None -> per-product frozen date; or e.g. '2026-08-01'
RUN_ID      = None        # None -> latest run per product; or a pinned run_id
PRODUCTS    = None        # None -> all trained products; or ['fx_activation', ...]

products = PRODUCTS or discover_products(ART_ROOT)
assert products, f'no trained models found under {ART_ROOT}'
print('scoring', len(products), 'products:', products)

### Score each product → write long format
One product is loaded, scored, and written at a time (partitioned by product; first write overwrites the path, the rest append), so the driver only ever holds one product's scores. A product without a usable model is skipped, not fatal — the batch continues.

In [ ]:
import pandas as pd
summary, first_write = [], True
for i, product in enumerate(products, 1):
    try:
        run_dir = resolve_run(product, ART_ROOT, run_id=RUN_ID)
        cfg, model, calibrate, model_hash = load_scorer(run_dir)
        idate = INFER_DATE or str(cfg.infer_date)
        long = score_product(spark, cfg, model, calibrate, idate, model_hash)
        sdf = spark.createDataFrame(long)                 # cols: cif, product, model_hash, infer_date, score
        (sdf.write.mode('overwrite' if first_write else 'append')
            .partitionBy('product').parquet(OUTPUT_PATH))
        first_write = False
        summary.append({'product': product, 'status': 'ok', 'run': run_dir.name,
                        'model_hash': model_hash, 'infer_date': idate, 'rows': len(long)})
        print(f'[{i}/{len(products)}] {product}: {len(long):,} rows')
    except Exception as e:
        summary.append({'product': product, 'status': f'SKIPPED: {e}', 'rows': 0})
        print(f'[{i}/{len(products)}] {product}: SKIPPED - {e}')

pd.DataFrame(summary)

### Verify the written output

In [ ]:
out = spark.read.parquet(OUTPUT_PATH)
print('total rows:', out.count())
out.groupBy('product').count().orderBy('product').show()
out.select('cif', 'product', 'model_hash', 'infer_date', 'score').show(5)

### Notes
- **Reproducible:** scoring uses the frozen config + saved model, so editing a product YAML later never changes what a given `model_hash` produces.
- **`model_hash`** is the content hash of that run's `model.pkl` (12 hex chars) — pins the exact model behind every score.
- **Monthly re-scoring:** set `INFER_DATE` to the new month and re-run; the same trained models score the fresh snapshot.
- **Scale:** each product's eligible population is pulled to pandas (pruned to features) for scoring; if one product's eligible set is very large, score it in Spark batches instead.